# Effect of Regularization Tolerance (rtol)

**Sweep:** `regularization`  
**Question:** How much GSVD regularization is needed? What happens to accuracy when rtol is too loose or too tight?

**Sweep variable:** `rtol` ∈ {1e-6, 1e-8, 1e-10, 1e-12, 1e-14, 1e-16}  
**Fixed:** `n_fb = 100`, `n_fs = 100`, `n_eigs = 10`

**Domains:** rect, L_shape, GWW1 (all have reference eigenvalues)

**Expected behaviour:** A U-shaped curve in error vs rtol on log-log axes — too loose (large rtol) truncates legitimate basis functions; too tight (small rtol) admits numerical noise into the GSVD nullspace.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import nb_utils

nb_utils.set_publication_style()
RESULTS_DIR = os.path.abspath('../results')

df, raw = nb_utils.load_sweep('regularization', RESULTS_DIR)
print(f'Loaded {len(df)} results')
print('Domains:', df['domain_name'].unique())
print('rtol values:', sorted(df['rtol'].unique()))
df.head(3)

## Plot 1: Accuracy vs Regularization Tolerance

The x-axis is ordered from *more* regularization (left, large rtol) to *less* (right, small rtol). The dashed vertical line marks the default value of `rtol = 1e-12`.

In [ ]:
domains = sorted(df['domain_name'].unique())
colors = nb_utils.domain_color_map(domains)

fig, ax = plt.subplots(figsize=(7, 4))
for dom in domains:
    d = df[df['domain_name'] == dom].sort_values('rtol')
    ax.loglog(d['rtol'], d['max_rel_error'],
              marker='o', color=colors[dom],
              label=nb_utils.label_domain(dom))

nb_utils.annotate_default_rtol(ax, 1e-12)
nb_utils.accuracy_threshold_line(ax, 1e-10, label='1e-10 accuracy')
ax.invert_xaxis()  # left = more regularisation
ax.set_xlabel('rtol  (← more regularisation          less regularisation →)')
ax.set_ylabel('Max relative eigenvalue error')
ax.set_title('Accuracy vs GSVD regularisation tolerance')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Plot 2: MPS Tension vs rtol

Does tension track relative error? If so, tension alone (without a reference) would reliably signal when regularisation is poorly tuned.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for dom in domains:
    d = df[df['domain_name'] == dom].sort_values('rtol')
    ax.loglog(d['rtol'], d['median_tension'],
              marker='o', color=colors[dom],
              label=nb_utils.label_domain(dom))

nb_utils.annotate_default_rtol(ax, 1e-12)
ax.invert_xaxis()
ax.set_xlabel('rtol  (← more regularisation          less regularisation →)')
ax.set_ylabel('Median MPS tension')
ax.set_title('MPS tension vs regularisation tolerance')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Plot 3: Per-Eigenvalue Error Heatmap

Which eigenvalues degrade first as rtol is pushed to an extreme? Each cell shows log₁₀(relative error).

In [ ]:
# index: (domain, rtol) → raw result
index = {}
for r in raw:
    index[(r.config.domain_name, r.config.rtol)] = r

rtol_vals = sorted(df['rtol'].unique())
fig, axes = plt.subplots(1, len(domains), figsize=(5 * len(domains), 4))
if len(domains) == 1:
    axes = [axes]

for ax, dom in zip(axes, domains):
    # Build matrix: rows = eigenvalue index, cols = rtol
    rows = []
    for rt in rtol_vals:
        r = index.get((dom, rt))
        if r is None or r.rel_errors is None:
            rows.append(np.full(10, np.nan))
        else:
            rows.append(np.log10(r.rel_errors + 1e-20))
    mat = np.array(rows).T  # shape: (n_eigs, n_rtol)

    im = ax.imshow(mat, aspect='auto', cmap='RdYlGn_r',
                   vmin=-15, vmax=0, origin='upper')
    ax.set_xticks(range(len(rtol_vals)))
    ax.set_xticklabels([f'{np.log10(rt):.0f}' for rt in rtol_vals], fontsize=8)
    ax.set_xlabel('log₁₀(rtol)', fontsize=9)
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels([str(i+1) for i in range(mat.shape[0])], fontsize=8)
    ax.set_ylabel('Eigenvalue index', fontsize=9)
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)
    fig.colorbar(im, ax=ax, label='log₁₀(rel. error)', fraction=0.04)

fig.suptitle('Per-eigenvalue relative errors by rtol', fontsize=11)
plt.tight_layout()
plt.show()

## Plot 4: Number of Eigenvalues Returned vs rtol

Does aggressive regularisation (large rtol) cause the solver to return fewer than the requested number of eigenvalues?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for dom in domains:
    d = df[df['domain_name'] == dom].sort_values('rtol')
    ax.semilogx(d['rtol'], d['n_eigs_returned'],
                marker='o', color=colors[dom],
                label=nb_utils.label_domain(dom))

nb_utils.annotate_default_rtol(ax, 1e-12)
ax.axhline(10, color='gray', linestyle='--', linewidth=0.8, label='Requested n_eigs=10')
ax.invert_xaxis()
ax.set_xlabel('rtol  (← more regularisation          less regularisation →)')
ax.set_ylabel('Eigenvalues returned')
ax.set_title('Number of eigenvalues returned vs rtol')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()